# Day 18 - Big Data Fundamentals: Delivery Bottleneck Hubs

**Program:** Capstone Anteraja - Minggu 4 (Data)
**Tools:** Google Colab + PySpark (Apache Spark 3.5)

Notebook ini menjawab pertanyaan:

> **Hub mana yang memiliki dwell time tertinggi dan perlu diinvestigasi lebih lanjut?**

Alurnya:

1. Setup SparkSession dan memuat dataset simulasi scan_events.
2. Memeriksa kualitas pasangan scan (schema, missing, duplicate, timestamp invalid, urutan terbalik).
3. Menghitung dwell time per package_id x hub_id dari pasangan **ARRIVAL -> DEPARTURE**.
4. Grouping, aggregate (count / average / median) dan sorting per hub.
5. Memilih 3-5 bottleneck hub dengan mempertimbangkan volume paket dan median.
6. Menulis satu paragraf business insight dari angka hasil analisis.

## 0. 5 Vs of Big Data dan keterbatasan Pandas

Big Data biasanya dijelaskan lewat lima karakteristik berikut:

| V | Arti | Contoh di dataset ini |
| --- | --- | --- |
| **Volume** | Ukuran data jauh melampaui memori satu mesin | Log scan dari seluruh hub terus bertambah tiap hari |
| **Velocity** | Data datang dan harus diproses cepat | Event scan masuk nyaris real-time dari scanner hub |
| **Variety** | Format dan sumber beragam | Event ARRIVAL/DEPARTURE, timestamp, plus data hub dan SLA |
| **Veracity** | Kualitas data tidak selalu terjamin | Scan hilang, duplikat, dan timestamp rusak (section 2) |
| **Value** | Nilai bisnis dari data | Menemukan hub bottleneck untuk prioritas investigasi |

**Kenapa bukan Pandas saja?** Pandas memuat seluruh data ke memori satu proses
(single machine), jalan di satu core, dan operasi groupby besar rawan kehabisan RAM.
Untuk jutaan sampai miliaran event scan, jawabannya adalah mesin terdistribusi seperti Spark:
data dipecah menjadi partisi, diproses paralel, dan agregasi dilakukan di banyak worker sekaligus.
Dataset di notebook ini sengaja berskala sedang agar bisa dijalankan di Colab gratis, tetapi pola
kodenya (filter -> group -> aggregate -> sort) langsung berlaku untuk skala produksi.

In [1]:
# Semua library di bawah tersedia di Google Colab. Kalau pyspark belum ada, blok ini
# memasangnya; kalau sudah ada, tidak ada proses install apa pun.
import os
import sys
import json
from pathlib import Path

try:
    import pyspark  # noqa: F401
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])

import numpy as np
import pandas as pd

# Driver dan worker Spark wajib memakai interpreter Python yang sama.
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

from pyspark.sql import SparkSession, functions as F, types as T

SEED = 42

# Notebook ini bisa dijalankan dari folder analysis/ (struktur repo) maupun dari
# folder datar seperti /content di Google Colab.
BASE = Path.cwd()
if (BASE / "analysis").is_dir():
    ANALYSIS_DIR, OUTPUT_DIR = BASE / "analysis", BASE / "output"
elif BASE.name == "analysis":
    ANALYSIS_DIR, OUTPUT_DIR = BASE, BASE.parent / "output"
else:
    ANALYSIS_DIR, OUTPUT_DIR = BASE, BASE / "output"
ANALYSIS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = ANALYSIS_DIR / "scan_events.csv"
OUTPUT_CSV = OUTPUT_DIR / "top_bottleneck_hubs.csv"
print("Analysis dir :", ANALYSIS_DIR.resolve())
print("Output dir   :", OUTPUT_DIR.resolve())

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day18-bottleneck-hubs")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Analysis dir : /private/tmp/bigdata-bottleneck-hubs/docs/big-data-bottleneck-analysis/analysis
Output dir   : /private/tmp/bigdata-bottleneck-hubs/docs/big-data-bottleneck-analysis/output


26/10/07 09:51:01 WARN Utils: Your hostname, kagchis-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.0.4 instead (on interface en0)
26/10/07 09:51:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/07 09:51:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3


## 1. Dataset simulasi scan_events

Schema minimal sesuai brief: hub_id | package_id | event_type | timestamp

Fungsi generate_scan_events() di bawah membuat dataset dengan seed tetap (**deterministik**),
jadi file yang di-commit dan hasilnya selalu sama. Setiap paket melewati 1-3 hub, dan untuk
setiap kunjungan dihasilkan pasangan event **ARRIVAL** lalu **DEPARTURE**.

Dataset sengaja dibuat kotor supaya section kualitas data benar-benar teruji:

- sebagian event ARRIVAL / DEPARTURE dihilangkan,
- beberapa scan digandakan (duplikat baris identik),
- sebagian timestamp diganti nilai yang tidak bisa di-parse,
- sebagian pasangan dibalik sehingga DEPARTURE jatuh sebelum ARRIVAL.

Kalau analysis/scan_events.csv sudah ada, file itu langsung dipakai; kalau belum, file dibuat
ulang secara identik.

In [2]:
HUB_PROFILE = {
    # hub_id: (rata-rata dwell dasar dalam jam, bobot volume paket)
    "HUB_CGK": (4.5, 14.0),
    "HUB_SBY": (6.4, 12.0),
    "HUB_BDG": (5.2, 11.0),
    "HUB_SMG": (5.8, 10.0),
    "HUB_JOG": (6.1, 9.0),
    "HUB_MKS": (10.4, 9.0),
    "HUB_MDN": (8.7, 8.0),
    "HUB_DPS": (9.3, 7.0),
    "HUB_PDG": (7.0, 6.0),
    "HUB_PLM": (7.6, 6.0),
    "HUB_BJM": (8.1, 5.0),
    "HUB_AMB": (11.2, 0.2),
}
N_PACKAGES = 13500
START = pd.Timestamp("2026-01-01 00:00:00")
SPAN_HOURS = 28 * 24  # paket mulai tersebar selama 28 hari
TS_FMT = "%Y-%m-%d %H:%M:%S"
BAD_TS = ["", "N/A", "not-a-date", "2026-02-30 25:61:00"]


def generate_scan_events(seed=SEED):
    rng = np.random.default_rng(seed)
    hubs = list(HUB_PROFILE)
    weights = np.array([HUB_PROFILE[h][1] for h in hubs], dtype=float)
    weights = weights / weights.sum()

    hops = []
    for i in range(N_PACKAGES):
        package_id = f"PKG{i + 1:06d}"
        n_hops = int(rng.choice([1, 2, 3], p=[0.45, 0.40, 0.15]))
        route = rng.choice(hubs, size=n_hops, replace=False, p=weights)
        t = START + pd.Timedelta(hours=float(rng.uniform(0, SPAN_HOURS)))
        for hub_id in route:
            base_dwell = HUB_PROFILE[hub_id][0]
            dwell = max(0.5, float(base_dwell * np.exp(rng.normal(0.0, 0.25))))
            arrival = t
            departure = arrival + pd.Timedelta(hours=dwell)
            hops.append((hub_id, package_id, arrival, departure))
            t = departure + pd.Timedelta(hours=float(rng.uniform(3, 12)))

    hop_df = pd.DataFrame(hops, columns=["hub_id", "package_id", "arrival", "departure"])

    # 1% pasangan dibalik supaya DEPARTURE jatuh sebelum ARRIVAL.
    reversed_mask = rng.random(len(hop_df)) < 0.01
    arrival_tmp = hop_df.loc[reversed_mask, "arrival"].copy()
    hop_df.loc[reversed_mask, "arrival"] = hop_df.loc[reversed_mask, "departure"]
    hop_df.loc[reversed_mask, "departure"] = arrival_tmp

    # Sebagian event sengaja hilang supaya ada paket tanpa ARRIVAL atau tanpa DEPARTURE.
    drop_arrival = rng.random(len(hop_df)) < 0.03
    drop_departure = rng.random(len(hop_df)) < 0.03

    rows = []
    for (hub_id, package_id, arrival, departure), da, dd in zip(
        hop_df.itertuples(index=False), drop_arrival, drop_departure
    ):
        if not da:
            rows.append((hub_id, package_id, "ARRIVAL", arrival))
        if not dd:
            rows.append((hub_id, package_id, "DEPARTURE", departure))

    events = pd.DataFrame(rows, columns=["hub_id", "package_id", "event_type", "timestamp"])

    # Semua timestamp valid diformat ke string lebih dulu (presisi detik),
    # baru sebagian diganti nilai rusak supaya kolomnya memang punya cacat.
    events["timestamp"] = pd.to_datetime(events["timestamp"]).dt.strftime(TS_FMT)
    bad_mask = rng.random(len(events)) < 0.01
    events.loc[bad_mask, "timestamp"] = rng.choice(BAD_TS, size=int(bad_mask.sum())).tolist()

    # Scan duplikat: baris yang benar-benar identik.
    dup_mask = rng.random(len(events)) < 0.02
    events = pd.concat([events, events[dup_mask]], ignore_index=True)
    events = events.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return events


if DATA_PATH.exists():
    print(f"Memuat dataset yang sudah ada: {DATA_PATH}")
else:
    scan_events_pd = generate_scan_events()
    scan_events_pd.to_csv(DATA_PATH, index=False)
    print(f"scan_events.csv dibuat ulang dengan seed {SEED}: {len(scan_events_pd):,} baris")

SCAN_SCHEMA = T.StructType([
    T.StructField("hub_id", T.StringType()),
    T.StructField("package_id", T.StringType()),
    T.StructField("event_type", T.StringType()),
    T.StructField("timestamp", T.StringType()),
])
raw = spark.read.csv(str(DATA_PATH), header=True, schema=SCAN_SCHEMA)
print("Dibaca Spark:", f"{raw.count():,}", "baris")

Memuat dataset yang sudah ada: /private/tmp/bigdata-bottleneck-hubs/docs/big-data-bottleneck-analysis/analysis/scan_events.csv


Dibaca Spark: 45,542 baris


In [3]:
raw.printSchema()
print("Total event :", f"{raw.count():,}")
print("Hub unik    :", raw.select("hub_id").distinct().count())
print("Paket unik  :", raw.select("package_id").distinct().count())
print("Jenis event :", [r.event_type for r in raw.select("event_type").distinct().orderBy("event_type").collect()])
raw.show(5, truncate=False)

root
 |-- hub_id: string (nullable = true)
 |-- package_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: string (nullable = true)

Total event : 45,542


Hub unik    : 12
Paket unik  : 13494


Jenis event : ['ARRIVAL', 'DEPARTURE']


+-------+----------+----------+-------------------+
|hub_id |package_id|event_type|timestamp          |
+-------+----------+----------+-------------------+
|HUB_MKS|PKG012292 |ARRIVAL   |2026-01-25 10:19:27|
|HUB_SBY|PKG001788 |ARRIVAL   |2026-01-08 13:59:20|
|HUB_DPS|PKG005129 |DEPARTURE |2026-01-23 12:59:26|
|HUB_SBY|PKG008499 |ARRIVAL   |2026-01-21 22:26:35|
|HUB_SBY|PKG003131 |DEPARTURE |2026-01-25 04:21:45|
+-------+----------+----------+-------------------+
only showing top 5 rows



## 2. Data quality check

Sebelum menghitung dwell time, pasangan scan diperiksa dulu. Pemeriksaan yang dilakukan:

1. **Schema & jumlah event** (section 1).
2. **Timestamp tidak valid** - string yang gagal di-parse to_timestamp() menjadi NULL.
3. **Scan duplikat** - baris identik pada hub_id, package_id, event_type, timestamp.
4. **Missing ARRIVAL / missing DEPARTURE** - pasangan package x hub yang kehilangan salah satu sisi.
5. **DEPARTURE sebelum ARRIVAL** - last_departure <= first_arrival, indikasi timestamp tertukar.

Hanya pasangan yang lolos semua pemeriksaan di atas yang dipakai untuk analisis berikutnya.

In [4]:
parsed = raw.withColumn("ts", F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))

total_rows = parsed.count()
invalid_ts = parsed.filter(F.col("ts").isNull()).count()

with_ts = parsed.filter(F.col("ts").isNotNull())
deduped = with_ts.dropDuplicates(["hub_id", "package_id", "event_type", "ts"])
duplicate_scans = with_ts.count() - deduped.count()

print(f"Total event mentah    : {total_rows:,}")
print(f"Timestamp tidak valid : {invalid_ts:,} ({invalid_ts / total_rows:.2%})")
print(f"Scan duplikat         : {duplicate_scans:,}")
print(f"Event bersih          : {deduped.count():,}")

Total event mentah    : 45,542
Timestamp tidak valid : 412 (0.90%)
Scan duplikat         : 881
Event bersih          : 44,249


In [5]:
grouped = (
    deduped.groupBy("hub_id", "package_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "ARRIVAL", 1).otherwise(0)).alias("n_arrival"),
        F.sum(F.when(F.col("event_type") == "DEPARTURE", 1).otherwise(0)).alias("n_departure"),
        F.min(F.when(F.col("event_type") == "ARRIVAL", F.col("ts"))).alias("first_arrival"),
        F.max(F.when(F.col("event_type") == "DEPARTURE", F.col("ts"))).alias("last_departure"),
    )
    .cache()
)

n_pairs = grouped.count()
missing_arrival = grouped.filter((F.col("n_departure") > 0) & (F.col("n_arrival") == 0)).count()
missing_departure = grouped.filter((F.col("n_arrival") > 0) & (F.col("n_departure") == 0)).count()
reversed_pairs = grouped.filter(
    (F.col("n_arrival") > 0)
    & (F.col("n_departure") > 0)
    & (F.col("last_departure") <= F.col("first_arrival"))
).count()

print(f"Pasangan package x hub : {n_pairs:,}")
print(f"Tanpa ARRIVAL          : {missing_arrival:,} ({missing_arrival / n_pairs:.2%})")
print(f"Tanpa DEPARTURE        : {missing_departure:,} ({missing_departure / n_pairs:.2%})")
print(f"DEPARTURE <= ARRIVAL   : {reversed_pairs:,} ({reversed_pairs / n_pairs:.2%})")

Pasangan package x hub : 22,991
Tanpa ARRIVAL          : 885 (3.85%)
Tanpa DEPARTURE        : 848 (3.69%)
DEPARTURE <= ARRIVAL   : 222 (0.97%)


In [6]:
quality = {
    "total_events": total_rows,
    "invalid_timestamps": invalid_ts,
    "duplicate_scans": duplicate_scans,
    "package_hub_pairs": n_pairs,
    "missing_arrival": missing_arrival,
    "missing_departure": missing_departure,
    "reversed_pairs": reversed_pairs,
}
print(json.dumps(quality, indent=2))

assert invalid_ts > 0, "Seharusnya ada timestamp tidak valid untuk dideteksi."
assert duplicate_scans > 0, "Seharusnya ada scan duplikat untuk dideteksi."
assert missing_arrival > 0 and missing_departure > 0 and reversed_pairs > 0
assert deduped.filter(F.col("ts").isNull()).count() == 0
print("\nValidasi kualitas lolos: semua jenis cacat terdeteksi, data bersih bebas duplikat dan timestamp invalid.")

{
  "total_events": 45542,
  "invalid_timestamps": 412,
  "duplicate_scans": 881,
  "package_hub_pairs": 22991,
  "missing_arrival": 885,
  "missing_departure": 848,
  "reversed_pairs": 222
}

Validasi kualitas lolos: semua jenis cacat terdeteksi, data bersih bebas duplikat dan timestamp invalid.


## 3. Dwell time per package_id x hub_id

Definisi yang dipakai, konsisten dengan brief:

- **ARRIVAL** = first_arrival = scan ARRIVAL paling awal pada pasangan tersebut.
- **DEPARTURE** = last_departure = scan DEPARTURE paling akhir pada pasangan tersebut.
- **Dwell Time (jam)** = (last_departure - first_arrival) / 3600.

Pasangan yang masih minus atau nol (penanda timestamp terbalik) sudah dibuang pada tahap
kualitas data, sehingga di sini semua nilai dwell dipastikan positif.

In [7]:
valid_pairs = (
    grouped.filter(
        (F.col("n_arrival") > 0)
        & (F.col("n_departure") > 0)
        & (F.col("last_departure") > F.col("first_arrival"))
    )
    .withColumn(
        "dwell_hours",
        (F.unix_timestamp("last_departure") - F.unix_timestamp("first_arrival")) / F.lit(3600.0),
    )
    .select("hub_id", "package_id", "first_arrival", "last_departure", "dwell_hours")
    .cache()
)

valid_count = valid_pairs.count()
print(f"Pasangan valid dipakai : {valid_count:,}")
print(f"Pasangan dibuang       : {n_pairs - valid_count:,}")
valid_pairs.select("hub_id", "package_id", "dwell_hours").show(5, truncate=False)

assert valid_pairs.filter(F.col("dwell_hours") <= 0).count() == 0, "Masih ada dwell time non-positif!"
print("Validasi lolos: semua dwell time positif.")

Pasangan valid dipakai : 21,036
Pasangan dibuang       : 1,955
+-------+----------+------------------+
|hub_id |package_id|dwell_hours       |
+-------+----------+------------------+
|HUB_PDG|PKG001460 |7.482777777777778 |
|HUB_SMG|PKG004272 |5.8902777777777775|
|HUB_CGK|PKG007865 |4.7219444444444445|
|HUB_CGK|PKG004472 |4.154722222222222 |
|HUB_JOG|PKG003414 |5.229444444444445 |
+-------+----------+------------------+
only showing top 5 rows

Validasi lolos: semua dwell time positif.


## 4. Aggregate per hub (filter -> group -> aggregate -> sort)

Sama seperti SQL yang sudah dipelajari, hanya mesinnya beda. Query di bawah menghitung
package_count, avg_dwell_hours, dan median_dwell_hours per hub_id, lalu mengurutkan dari
average tertinggi.

In [8]:
valid_pairs.createOrReplaceTempView("valid_pairs")

hub_dwell = spark.sql("""
    SELECT
        hub_id,
        COUNT(*) AS package_count,
        AVG(dwell_hours) AS avg_dwell_hours,
        percentile_approx(dwell_hours, 0.5) AS median_dwell_hours
    FROM valid_pairs
    GROUP BY hub_id
    ORDER BY avg_dwell_hours DESC
""")

# Hasil agregasi per hub hanya berisi belasan baris, jadi dikumpulkan ke driver
# lalu dirapikan dengan pandas (tidak memakai toPandas agar tidak bergantung distutils).
hub_dwell_pd = pd.DataFrame([row.asDict() for row in hub_dwell.collect()]).round(2)
print(hub_dwell_pd.to_string(index=False))

 hub_id  package_count  avg_dwell_hours  median_dwell_hours
HUB_AMB             36            12.32               11.71
HUB_MKS           1928            10.70               10.30
HUB_DPS           1580             9.48                9.16
HUB_MDN           1702             8.96                8.59
HUB_BJM           1096             8.38                8.20
HUB_PLM           1349             7.82                7.56
HUB_PDG           1357             7.34                7.17
HUB_SBY           2545             6.59                6.37
HUB_JOG           1969             6.24                6.05
HUB_SMG           2197             6.01                5.79
HUB_BDG           2324             5.33                5.18
HUB_CGK           2953             4.62                4.46


## 5. Identifikasi bottleneck hub

Average saja bisa menyesatkan: hub dengan sedikit paket gampang terlihat ekstrem. Karena itu
ranking memakai **ambang volume minimum** supaya hub kecil tidak otomatis masuk prioritas,
sekaligus membandingkan **median** untuk memastikan bukan hanya segelintir paket yang lama.

In [9]:
MIN_PACKAGES = 100

ranked = hub_dwell_pd.sort_values("avg_dwell_hours", ascending=False).reset_index(drop=True)
eligible = ranked[ranked["package_count"] >= MIN_PACKAGES].reset_index(drop=True)
excluded = ranked[ranked["package_count"] < MIN_PACKAGES].reset_index(drop=True)
top_hubs = eligible.head(5).reset_index(drop=True)

top_hubs.to_csv(OUTPUT_CSV, index=False)

print(f"Ambang minimum paket per hub: {MIN_PACKAGES}")
if len(excluded):
    print("Hub dikecualikan karena volume paket terlalu kecil:")
    print(excluded.to_string(index=False))
print(f"\nTop {len(top_hubs)} bottleneck hub (disimpan ke {OUTPUT_CSV.name}):")
print(top_hubs.to_string(index=False))

assert 3 <= len(top_hubs) <= 5, "Jumlah hub bottleneck harus 3-5."
assert OUTPUT_CSV.exists(), "CSV gagal dibuat."
assert list(top_hubs.columns) == ["hub_id", "package_count", "avg_dwell_hours", "median_dwell_hours"]

summary = {
    "quality": quality,
    "valid_pairs": valid_count,
    "min_packages": MIN_PACKAGES,
    "excluded": excluded.to_dict("records"),
    "top_hubs": top_hubs.to_dict("records"),
    "overall_avg_dwell_hours": round(float(valid_pairs.select(F.avg("dwell_hours")).first()[0]), 2),
}
print("SUMMARY_JSON:" + json.dumps(summary, default=float))

Ambang minimum paket per hub: 100
Hub dikecualikan karena volume paket terlalu kecil:
 hub_id  package_count  avg_dwell_hours  median_dwell_hours
HUB_AMB             36            12.32               11.71

Top 5 bottleneck hub (disimpan ke top_bottleneck_hubs.csv):
 hub_id  package_count  avg_dwell_hours  median_dwell_hours
HUB_MKS           1928            10.70               10.30
HUB_DPS           1580             9.48                9.16
HUB_MDN           1702             8.96                8.59
HUB_BJM           1096             8.38                8.20
HUB_PLM           1349             7.82                7.56
SUMMARY_JSON:{"quality": {"total_events": 45542, "invalid_timestamps": 412, "duplicate_scans": 881, "package_hub_pairs": 22991, "missing_arrival": 885, "missing_departure": 848, "reversed_pairs": 222}, "valid_pairs": 21036, "min_packages": 100, "excluded": [{"hub_id": "HUB_AMB", "package_count": 36, "avg_dwell_hours": 12.32, "median_dwell_hours": 11.71}], "top_hubs": [{"

## 6. Business Insight

> **HUB_MKS adalah bottleneck terbesar.** Dengan 1.928 pasangan package x hub yang valid,
> HUB_MKS mencatat average dwell time 10,70 jam dan median 10,30 jam, jauh di atas rata-rata
> keseluruhan 7,10 jam. Karena average dan median-nya berdekatan, lambatnya bukan efek
> segelintir paket ekstrem, melainkan pola yang merata. HUB_DPS (9,48 / 9,16 jam, 1.580 paket),
> HUB_MDN (8,96 / 8,59 jam, 1.702 paket), HUB_BJM (8,38 / 8,20 jam, 1.096 paket), dan
> HUB_PLM (7,82 / 7,56 jam, 1.349 paket) mengikuti dengan pola serupa. HUB_AMB sebenarnya punya
> average tertinggi (12,32 jam), tetapi hanya 36 paket valid sehingga dikecualikan oleh ambang
> minimum 100 paket. Sisi kualitas data juga perlu dibaca: dari 22.991 pasangan package x hub,
> 885 (3,85%) kehilangan ARRIVAL, 848 (3,69%) kehilangan DEPARTURE, 222 (0,97%) punya
> DEPARTURE <= ARRIVAL, 412 event bertimestamp rusak, dan 881 scan duplikat; setelah dibersihkan
> 21.036 pasangan (91,5%) dipakai untuk analisis. Rekomendasi investigasi: prioritaskan HUB_MKS,
> lalu HUB_DPS dan HUB_MDN, dengan memeriksa kapasitas dan alur sorting, jumlah dock/gate,
> pola shift pada jam sibuk, serta konsistensi pencatatan scan ARRIVAL-DEPARTURE.

## 7. Ringkasan Validasi

- Dataset scan_events dibaca Spark dengan schema eksplisit hub_id, package_id, event_type, timestamp.
- Pemeriksaan kualitas data mendeteksi timestamp tidak valid, scan duplikat, missing ARRIVAL,
  missing DEPARTURE, dan pasangan DEPARTURE <= ARRIVAL; hanya pasangan valid yang dianalisis.
- Dwell time dihitung dari pasangan ARRIVAL -> DEPARTURE dan dikonversi ke jam.
- Aggregate per hub menghasilkan package_count, avg_dwell_hours, dan median_dwell_hours,
  diurutkan dari average tertinggi.
- output/top_bottleneck_hubs.csv berisi 3-5 hub dengan rata-rata dwell time tertinggi
  (setelah ambang volume minimum).
- Business insight ditulis dari angka hasil analisis di atas.